# သင်ခန်းစာ 18: ကရစ်ပတိုဂရပ်ဖစ် လက်မှတ်များဖြင့် AI အေးဂျင့်များကို တပ်ဆင်ကာကွယ်ခြင်း

## လက်တွေ့လုပ်ဆောင်ချက် စာအုပ်

ဤစာအုပ်တွင် လေ့ကျင့်ခန်းလေးခုကို လမ်းညွှန်ပြသပါသည်။

1. **သင့်ရဲ့ ပထမဆုံး လက်မှတ်ကို လက်မှတ်ရေးထိုးခြင်း** အေးဂျင့်ကိရိယာခေါ်ဆိုမှုအတွက် နှင့် စစ်ဆေးခြင်း။
2. **လက်မှတ်ကို ပြုပြင်ဖျက်သိမ်းခြင်း** နှင့် စစ်ဆေးမှု မအောင်မြင်ခြင်းကို ကြည့်ရှုခြင်း။
3. **လက်မှတ်သုံးခုကို ကွက်တိကြိုးတစ်ခုအဖြစ် ဖွဲ့စည်းခြင်း** နှင့် ကြိုးတင်ပုံစံသက်ဆိုင်မှုကို အတည်ပြုခြင်း။
4. **Microsoft Agent Framework ကိရိယာခေါ်ဆိုမှုကို အမူအရာတိုင်းတွင် လက်မှတ် ထုတ်ပေးသည့် လုပ်ဆောင်ချက်အဖြစ် ဖု wrap လုပ်ခြင်း**။

ကရစ်ပတိုဂရပ်ဖစ်အခြေခံဆန်းသစ်တီထွင်မှုအားလုံးကို ကောင်းမွန်စွာ ထိန်းသိမ်းထားသောစာကြည့်တိုက်များမှ တင်သွင်းထားသည် (`pynacl` ကို Ed25519 အတွက်၊ `jcs` ကို RFC 8785 canonical JSON အတွက်၊ Python စံနမူနာစာကြည့်တိုက်မှ `hashlib` ကို SHA-256 အတွက်)။ လက်မှတ်လုပ်ငန်းစဉ်သည် မူရင်း Python ဖြစ်ပြီး သင်ဖတ်ရှု၍ ပြင်ဆင်နိုင်သည်။

ဆဲလ်များကို အစဉ်လိုက် ရိုက်ထည့်ပါ။ အပိုင်းတိုင်းသည်တိုတောင်းပြီး မိမိတိုင် မဟုတ်ဘဲ လုပ်ဆောင်နိုင်သည်။


## သတ်မှတ်တပ်ဆင်ခြင်း

အကြောင်းပြချက်နှစ်ခုကို တပ်ဆင်ပါ။ နှစ်ခုလုံးမှာ ခွင့်ပြုချက်ကောင်းစွာရှိသည် (Apache-2.0 / MIT)။


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## အကူအညီ အသုံးဝင် စက်ပစ္စည်းများ

ဒီအကူအညီ၂ခုက base64url ကုတ်ပြုခြင်း( padding မပါ) နဲ့ SHA-256 ဟက်ရှ်လုပ်ခြင်းကို ကိုင်တွယ်ပေးနေပြီး မှတ်ခွဲစာအုပ်ကို ငွေတောင်းလွှာအကြောင်း အာရုံစိုက်စေပါတယ်။


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## အပိုင်း ၁: သင့်ပထမဆုံး လက်မှတ်မိန့်ချက်ကို အတည်ပြုပါ

**Contoso Travel** ၏ ကိုယ်စားလှယ်သည် စီဒနွီးမှ လော့စ်အန်ဂျယ်လိပ်သို့ ဧည့်သည်တစ်ဦးအတွက် လေယာဉ်ခရီးစဉ်ရှာဖွေခဲ့သည်ဆိုပါစို့။ ကျွန်ုပ်တို့သည် အနာဂတ်တွင်စာရင်းစစ်သူတစ်ဦးက ကျွန်ုပ်တို့ကို ဘာမှမယုံကြည်ဘဲ အတည်ပြုနိုင်ရန် ဒီအကိရိယာခေါ်ဆိုမှုကို လက်မှတ်တင်ထားသော လက်မှတ်အဖြစ် မှတ်တမ်းတင်လိုသည်။

### အဆင့် ၁.၁: လက်မှတ်ရေးထိုး key တစ်ခု ကို ဖန်တီးပါ

ထုတ်လုပ်မှုတွင် ကိုယ်စားလှယ်၏ လက်မှတ်ရေးထိုး key ကို hardware security module (HSM), Azure Key Vault သို့မဟုတ် ဆင်တူသော ကာကွယ်ထားသော စတိုးဆိုင်ရာတွင် သိမ်းဆည်းထားမည် ဖြစ်သည်။ ဒီသင်ခန်းစာအတွက် ကျွန်ုပ်တို့သည် ဤ key ကို အမှတ်တရ၌ အသစ်တစ်ခု ဖန်တီးပါမည်။


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### အဆင့် ၁.၂: စာရင်းယူငွေပေးချေမှု ပေလုတ်တည်ဆောက်ခြင်း

ပေလုတ်တွင် စာရင်းယူငွေပေးချေမှုက အတည်ပြုချင်သမျှအားလုံးပါဝင်သည်။ ဘယ်သူလုပ်ဆောင်ခဲ့သည်၊ ဘယ်ကိရိယာကို အသုံးပြုခဲ့သလဲ၊ ဘာ အချက်အလက်များဖြင့်၊ ဘာတွေ ပြန်လာခဲ့သည်၊ ဘယ်မူဝါဒအောက်တွင် ဖြစ်သလဲ၊ နှင့် ဘယ်အချိန်တွင် ဖြစ်သလဲ စသည်တို့ဖြစ်သည်။ အဆိုပါအချက်အလက်များနှင့်ရလဒ်ကို တန်းထဲထည့်ခြင်းမပြုဘဲ အစား hash ဖတ်၍ စာရင်းယူငွေပေးချေမှုသည် လျှို့ဝှက်သော အကြောင်းအရာများ မပြန့်ပွားစေရန်ဖြစ်သည်။


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### ခြေလှမ်း ၁.၃: လက်မှတ်ရေးထိုးပြီး ချမှတ်ချက်ကိုစုစည်းပါ

တစ်ချက်သုံးချက်များ-

၁။ ကြေညာချက်ကို JCS ကိုအသုံးပြုပြီး canonicalize လုပ်ပါ၊ ထို့ကြောင့် အတူတူ အလားတူစာချုပ်ကိုထုတ်လုပ်သော implementation နှစ်ခုသည် byte များစွာလုံးဝတူညီသော bytes များထုတ်ပေးနိုင်ပါသည်။
၂။ Canonical bytes များကို SHA-256 ဖြင့် hash လုပ်ပါ။
၃။ Hash ကို Ed25519 ကိုယ်ပိုင် key ဖြင့် လက်မှတ်ရေးထိုးပါ။

လက်မှတ်အားမူရင်း ကြေညာချက်တွင် ချိတ်ဆက်ပြီး အဆုံးစွန် receipt ကိုထုတ်ပေးပါသည်။


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### အဆင့် 1.4: လက်မှတ်ခံလက်မှတ်မှန်ကန်မှု စစ်ဆေးခြင်း

စစ်ဆေးမှုသည် လုပ်ငန်းစဉ်ကို မူလပြန်လည်ပြောင်းရွှေ့ခြင်း ဖြစ်သည်။ ကျွန်ုပ်တို့သည် လက်မှတ်ကိုဖယ်ရှားပြီး၊ canonical hash ကို ထပ်မံတွက်ချက်ပြီး၊ လက်မှတ်ကို လက်ခံရရှိသော အများပြည်သူသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသောသော,မှန်ကန်မှုစစ်ဆေးသည်။

ဒီ စစ်ဆေးမှုကိုပြုလုပ်နေတဲ့ စစ်ဆေးသူတစ်ဦးက ကျွန်တော်တို့ဆီကနေ လိုအပ်တာက ရှိတာက လက်မှတ်ပဲဖြစ်ပါတယ်။ ဆာဗစ်စ်က သုံးရန်မလိုဘဲ၊ key directory ကို ရှာဖွေရန်မလိုဘဲ၊ ယုံကြည်မှုမလိုဘဲ။


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

သင်သည် `Receipt is valid: True` ကို ညွှန်ပြသင့်သည်။ အဲဂျင့်ကို ၎င်း၏ ပထမဆုံး သို့ဂရပ်ဖ်ဟိုးလ်လ်အတည်ပြုချက်ဖြင့် လက်မှတ်ရေးထိုးထားသော စန်းစစ်မှတ်တမ်း ထုတ်လုပ်ထားသည်။


## အပိုင်း ၂: လက်ခံရွတ်တမ်းကို ချွတ်မှားခြင်း

လက်ခံရွတ်တမ်းများ၏ အဓိကရည်ရွယ်ချက်မှာ ၎င်းတို့သည် ချွတ်မှားမှုများကို ရိုက်ချက်ပေးနိုင်ဖို့ ဖြစ်သည်။ ထိုအချက်ကို သက်သေပြကြရအောင်။

လက်ခံရွတ်တမ်းမှတစ်လုံးလုံး သတ်မှတ်ထားသော တစ်လုံးတည်းသော စာလုံးကို ပြင်ဆင်ပြီး အတည်ပြုမှု မအောင်မြင်မှုကို ကြည့်မည်။


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### ဘာဖြစ်ခဲ့တာလဲ?

ကျွန်ုပ်တို့ `policy_id` ကိုပြောင်းလဲခဲ့တဲ့အခါ canonical bytes များပြောင်းလဲသွားသည်။ ထို bytes များ၏ SHA-256 hash အလည်းပြောင်းလဲသွားသည်။ လက်မှတ် (မူလ hash ဖြင့်စမ်းသပ်ထားသော) သည် hash အသစ်နှင့်မကိုက်ညီတော့ပါ။ သေချာစစ်ဆေးမှုသည် မှန်ကန်စွာ `False` ပြန်ထုတ်သည်။

လက်မှတ်ရရှိမှုကို သက်ဆိုင်ရာ field များကို မြစ်ပြောင်း၍ ထိန်းညှိလို့မရနိုင်ပါ၊ သို့မဟုတ် တိုးဖောက်သူမှာ private key ကိုသာပိုင်ဆိုင်ထားရမည်ဖြစ်သည်။ private key ကို key vault ထဲမှာထားပြီး public key ကို စာရင်းထဲမှာ ထုတ်ဖော်ထားမှ tampering ကိုလျှိုဖို့မဖြစ်နိုင်ပါ။

ကိုယ်တိုင်ကြိုးစားကြည့်ပါ: အထက်ပါ ဥပမာမှ `tool_name` သို့မဟုတ် `agent_id` သို့မဟုတ် `timestamp` ကို ပြောင်းလဲပြီး ပြန်လည်ပြေးပါ။ ပြောင်းလဲမှုတိုင်းသည် မမှန်ကန်သော receipt ကိုဖန်တီးပေးသည်။


## အပိုင်း ၃: ရရှိမှုစာရွက်များကို စက်ဝိုင်းချိတ်ဆက်ပါ

တစ်ခုတည်းသော ရရှိမှုစာရွက်သည် လှုပ်ရှားမှုတစ်ခုကို ကာကွယ်ပေးသည်။ အေဂျင့်များအများစုသည် လှုပ်ရှားမှုများစွာလုပ်ဆောင်ကြသည်။ အစဉ်အဆက်တစ်ခုလုံးကို လှောင်ခြင်းအမှတ်အသား သက်သေအောင်လုပ်ရန်၊ ကျွန်ုပ်တို့သည် ယခင်ရရှိမှုစာရွက်၏ ဟက်ရှ်ကို အသစ်ရရှိမှုစာရွက်၏ ပေးလ်လိုက်ထဲတွင် ထည့်သွင်းခြင်းဖြင့် ရရှိမှုစာရွက်တစ်စိတ်တစ်ပိုင်းစီကို ယခင်ရရှိမှုနဲ့ချိတ်ဆက်ထားသည်။

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

မည်သူမဆို ရရှိမှုစာရွက်ကို ဖယ်ရှား하거나 အဆင့်အလိုက် ပြန်လည်စီစဉ်လျှင်၊ စက်ဝိုင်းသည် အတိအကျသော အချက်မှာ ချိုးကွက်သွားခဲ့သည်။ နောက်ပိုင်းရရှိမှုစာရွက်တစ်စိတ်တစ်ပိုင်းကို အတည်ပြုရာတွင် မအောင်မြင်ပါ၊ ကအကြောင်းမှာ ၎င်း၏ `previous_receipt_hash` သည် ယခင်ရရှိမှုစာရွက်၏ တကယ့်ဟက်ရှ်နှင့် ကိုက်ညီမှုမရှိတော့ဘူး။


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

အလယ်မှာရှိတဲ့ လက်ခံပေးမှုကို ပြောင်းပြန်ပြီး ထပ်မံစစ်ဆေး၍ အချိုးအတန်း ပြတ်တောက်စေပါ။ ပြောင်းလဲထားသော လက်ခံပေးမှုသည် ၎င်း၏ လက်မှတ်စစ်ဆေးမှုကို မဖြတ်နိုင်သေးဘဲ၊ နောက်တစ်ခုသော လက်ခံပေးမှုသည် ၎င်း၏ `previous_receipt_hash` သည် ပြောင်းလဲထားသည့် အလယ်လက်ခံမှု၏ hash နှင့် မကိုက်ညီတော့သောကြောင့် ချိတ်ဆက်မှုစစ်ဆေးမှုကို မဖြတ်နိုင်ပါ။


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 သည် (ပြင်ဆင်မှုမရှိသလို မူလအရင်းအမြစ်အပေါ်မူတည်မှုမရှိသောကြောင့်) မှန်ကန်မှုစစ်ဆေးရာမှာ အတည်ပြုထားဆဲဖြစ်သည်။ Receipt 1 သည် `tool_args_hash` ကိုပြောင်းလဲထားသောကြောင့် အမှတ်အသားစစ်ဆေးမှုတွင် ကျတာဖြစ်သည်။ Receipt 2 သည် မူရင်း (ယခုပြင်ဆင်ပြီးသော) receipt 1 ကို သုံး၍ ချိတ်ဆက်မှုစစ်ဆေးရာတွင် မအောင်မြင်သဖြင့် ကျတာဖြစ်သည်။

တိုက်ခိုက်သူသည် private key မရှိဘဲ ပြင်ဆင်ပြီးသော receipt 1 ကို ထပ်မံလက်မှတ်ရေးထိုး၍ မရနိုင်သော်လည်း receipt 2 မှာ ချိတ်ဆက်မှု မကိုက်ညီမှုကြောင့် အတုအယောင်ကို တွေ့ရှိနိုင်နေဆဲဖြစ်သည်။ ပြောင်းလဲမှုကို ဖုံးကွယ်ရန် တိုက်ခိုက်သူသည် ပြင်ဆင်မှုမှစ၍ Receipt အားလုံးကို ပြန်လက်မှတ်ရေးထိုးရမည်ဖြစ်ပြီး ၎င်းသည် private key ကိုပိုင်ဆိုင်ထားမှသာဖြစ်နိုင်သည်။


## အပိုင်း ၄: လူကိုယ်စားပြုကိရိယာခေါ်ဆိုမှုကို စာရင်းပေးသက်သေထိုးမှုဖြင့် ဝတ်ဆင်ခြယ်မှုလုပ်ခြင်း

အမှန်တကယ် တပ်ဆင်သည့်အခါ၊ လူကိုယ်စားပြုရေးသူတိုင်းသည် `make_receipt` ကို ခေါ်ရန် မှတ်မိရန် မလိုချင်ပါ။ ကိရိယာခေါ်ဆိုမှုတိုင်းမှာ စာရင်းပေးသက်သေထိုးမှု အလိုအလျောက် ဖြစ်စေရန်လိုအပ်ပါသည်။

ဒီမှာ အလွန်ရိုးရှင်းဆုံး ပုံစံတစ်ခုရှိသည်- မည်သည့်ခေါ်နိုင်သော ကိရိယာလုပ်ဆောင်ချက်မဆို ယူပြီး စာရင်းပေးသက်သေ ထုတ်ယူနိုင်သော ဗားရှင်းတစ်ခုကို ပြန်လည်ထုတ်ပေးသော ဝတ်ဆင်စားပုံမျိုး။ ၎င်းသည် Microsoft Agent Framework (`agent_framework.foundry`) အပါအဝင် မည်သည့် agent framework မဆို ကိုက်ညီနိုင်သည်။

သင်မှာ Microsoft Foundry ပရောဂျက် တစ်ခုလည်းမရှိပါက၊ ေအာက္တွင် ဒေသန္တရ mock သည် ထိုပုံစံကို ပြသပေးသည်။


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Microsoft Agent Framework နှင့် ပေါင်းစည်းခြင်း

အထက်က `ReceiptedTool` wrapper သည် framework-agnostic ဖြစ်သည်။ ထို wrapper function ကို Microsoft Agent Framework ဖြင့် တည်ဆောက်ထားသော agent အတွင်းတွင် အသုံးပြုရန်အတွက်၊ အဆိုပါ function ကို tool အဖြစ် မှတ်ပုံတင်ပါ။ ဤမှာ တစ်ခုခု ရေးဆွဲနေသော ဥပမာတစ်ခုဖြစ်ပြီး မော့ခ်ကို Microsoft Foundry ရဲ့ စစ်မှန်သော tool မှတ်ပုံတင်မှုဖြင့် အစားထိုးနိုင်သည်။

```python
# ပေါင်းစပ်မှု ပုံတူကို ပြသသော ပseudocode။
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="သင်သည် Contoso Travel အေဂျင့်တစ်ယောက်ဖြစ်သည် ...",
#     tools=[receipted_lookup],   # လှုပ်တင်ထားသော ကိရိယာ၊ မူလ function မဟုတ်ပါ
# )
# response = agent.run("ဇွန်လတွင် Sydney မှ Los Angeles သို့ အလေ့အပြတ်များ ရှာပါ။")
#
# # run ပြီးလျှင်၊ agent က သုံးသော tool call တစ်ခုစီမှာ လက်မှတ်ထိုးထားသော လက်မှတ် ရှိသည်။
# audit_chain = receipted_lookup.receipts
```

agent framework သည် receipt များနှင့် ပတ်သက်၍ မည်သည့် သတင်းအချက်အလက်ကိုမျှ မလိုအပ်ပါ။ Receipt မှတ်ချက်ရေးသားခြင်းကို tool အတွက် အဝိုင်းပတ်ထုပ်ပိုးထားပြီး framework ထဲတွင် မထည့်သွင်းထားပါ။  ဤနည်းလမ်းဖြင့် agent ကို ပြန်ရေးသားစရာမလိုဘဲ လက်ရှိရှိသော agent ကုဒ်များတွင် provenance ကို ထည့်သွင်းနိုင်သည်။


## ပြန်လည်သုံးသပ်ခြင်းနှင့် ကြိုးပမ်းစမ်းသပ်မှု

သင်သည်:

- Ed25519key ဇူးစုံတစ်စုံကို ဖန်တီးခဲ့သည်။
- အေးဂျင့်ခလုတ္Toolsခေါ်ဆိုမှုအတွက်လက်မှတ်ထိုးပြီး လက်ခံလက်မှတ်တစ်စောင်ကို ဆောက်လုပ်ခဲ့သည်။
- လက်မှတ်ကို ပုဂ္ဂိုလ်ရေးသော public key ဖြင့် အွန်လိုင်းမသုံးဘဲ ပြန်လည်စစ်ဆေးခဲ့သည်။
- လက်မှတ်တစ်စောင်ကို ခိုးကူးပြီး စစ်ဆေးမှု မအောင်မြင်မှုကို ကြည့်ရှူခဲ့သည်။
- လက်မှတ်သုံးစီးတာစဉ်ကွဲချားမှု၊ hash-chained sequence တစ်ခုကို ဆောက်လုပ်ခဲ့သည်။
- ကွဲချားမှုအလယ်တွင် ခိုးကူးပြီး လက်မှတ်အောင်မမြင်မှုနှင့် စီးတာချိတ်ဆှဲမှု မအောင်မြင်မှုနှစ်ခုစလုံးကို ကြည့်ရှူခဲ့သည်။
- Tool function တစ်ခုကို အလိုအလျောက် လက်မှတ်ထိုးရန် wrap လုပ်ခဲ့သည်။

**ကြိုးပမ်းစမ်းသပ်မှု။** `request_id` စံနှုန်း (အရေးအသားဖြန့်ဝေမှုအတွက် UUID) ပါဝင်သော receipt schema ကိုတိုးချဲ့ပါ။ `make_receipt` ကို အဲဒါပါဝင်ရန် ပြင်ဆင်ပြီး အချက်တင်လက်မှတ်များကိုအစမှာမှပါအောင်စစ်ဆေးနိုင်မှုရှိကြောင်း အတည်ပြုပါ။ ထို့နောက် လက်မှတ်ခိုးပြီး အသင်းထပ်လွှတ်ကန့်ရပ်၍ အတည်ပြုမှု မအောင်မြင်ကြောင်းပြင်၍ စစ်ဆေးပါ။ ၎င်းသည် canonical encoding ၏တိုင်း byte တစ်ခုချင်းစီသည် လက်မှတ်တွင် ဘယ်လိုထည့်သွင်းစေရန်ကို သင်မြင်နိုင်သည့်နည်းဖြစ်သည်။

**အရေးပါသော အကန့်အသတ်။** လက်မှတ်များသည် သုံးခုသာသက်သေပြသည်- အသီးသီး (ဤ key သည် ဤအကြောင်းအရာကို လက်မှတ်ထိုးသည်), သက်မှားခြင်း (အကြောင်းအရာသည် လက်မှတ်ထိုးပြီးမှ မပြောင်းလဲထားပါ), နှင့် စဉ်ပြက်ခြင်း (ဤလက်မှတ်သည် အဲ့လက်မှတ်နောက်တွင် လာသည်)။ ၎င်းသည် အေးဂျင့်၏ အပြုအမှု မှန်ကန်မှုကို သက်သေပြခြင်း မဟုတ်၊ `policy_id` တွင် အမည်ပေးထားသော မူဝါဒကို စစ်ဆေးခဲ့ကြောင်း သက်သေပြခြင်း မဟုတ်၊ သို့မဟုတ် အေးဂျင့်သည် စည်းကမ်းအားလုံးလိုက်နာကြောင်း သက်သေပြခြင်း မဟုတ်ပါ။ လက်မှတ်များသည် အခြေခံဖြစ်သည်။ အုပ်ချုပ်မှုသည် သင်တည်ဆောက်မည့် စနစ်ဖြစ်သည်။

အဆိုပါ အကန့်အသတ်ကို မှတ်သားပြီး သင်ခန်းစာ README ကို ထပ်ဖတ်ပါ။ လက်မှတ်များနှင့် ပတ်သက်၍ အဖွဲ့များ အများဆုံးမှားယွင်းခြင်းမှာ "ကျွန်ုပ်တို့မှာ လက်မှတ်များရှိသည်" ဆိုသည်မှာ "ကျွန်ုပ်တို့ အုပ်ချုပ်မှုရှိသည်" ဟူသော မှတ်ယူချက်ဖြစ်သည်။ ၎င်းမှာ မဟုတ်ပါ။ လက်မှတ်များသည် အေးဂျင့်၏ အပြုအမှုကို စစ်ဆေးနိုင်စေသည်။ ၎င်းသည် မှန်ကန်စေရန် မဟုတ်ပါ။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
